# Populate HexMazeThetaV1 for all hex maze sessions

Runs the full LFP to theta pipeline for all sessions that have ephys data and exist in `HexMazeBlock`, then populates `HexMazeThetaV1`.

Pipeline steps (handled automatically by `setup_theta_pipeline`):
1. LFP lowpass filter matched to the session's actual raw sampling rate (fetched from the database)
2. LFP electrode group (all electrodes from sort groups, excluding bad channels)
3. LFP filtered and downsampled to ~1000 Hz (`LFPV1`) — **~2 hours per session**
4. Theta bandpass filter (5–11 Hz) matched to actual LFP output rate
5. Theta-band LFP (`LFPBandV1`)
6. Analytic signal, phase, and power saved to NWB (`HexMazeThetaV1`)

In [1]:
import datajoint as dj
import numpy as np
import pandas as pd

import spyglass.common as sgc
import spyglass.lfp as lfp
import spyglass.spikesorting.v1 as sgs
# Berke lab IM-* sessions were sorted with spikesorting v1
# Frank lab sessions (e.g. Toby) have sort groups in spikesorting v0
from spyglass.spikesorting.v0.spikesorting_recording import SortGroup as SortGroupV0
from spyglass.lfp.analysis.v1.lfp_band import LFPBandV1, LFPBandSelection

from spyglass_hexmaze.hex_maze_behavior import HexMazeBlock
from spyglass_hexmaze.hex_maze_decoding import HexMazeThetaV1

/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-07-27 19:57:32,333][INFO]: DataJoint 0.14.6 connected to scrater@lmf-db.cin.ucsf.edu:3306
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-fiber-photometry': 
 ndx-fiber-photometry defines OpticalFiber.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines ExcitationSource.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines Photodetector.model as an attribute (dtype: text) while the core 

## Check current state of tables

In [2]:
# Pick the right SortGroup table for a session based on its nwb file name
# Berke lab IM-* sessions live in spikesorting v1 (sgs.SortGroup)
# Frank lab sessions (e.g. Toby) live in spikesorting v0 (SortGroupV0)
def sort_group_table(nwb_file_name):
    """Return the SortGroup table (v1 or v0) that holds this session's sort groups."""
    return sgs.SortGroup if nwb_file_name.startswith("IM-") else SortGroupV0


# The run-epoch interval(s) to process for a session.
# Berke lab IM-* sessions have a single run epoch: "00_r1"
# Frank lab sessions have 4 or 5 run epochs:
# epoch 1: "01_r1", epoch 3: "03_r2", epoch 5: "05_r3", epoch 7: "07_r4", epoch 9: "09_r5"
# In both cases we just look up the run interval of every epoch that has a HexMazeBlock from TaskEpoch
def get_run_intervals(nwb_file_name):
    """Return the run-epoch interval_list_name(s) that contain hex maze blocks."""
    hex_epochs = sorted(set((HexMazeBlock & {"nwb_file_name": nwb_file_name}).fetch("epoch")))
    return [
        (sgc.TaskEpoch & {"nwb_file_name": nwb_file_name, "epoch": e}).fetch1("interval_list_name")
        for e in hex_epochs
    ]


# All sessions in HexMazeBlock that also have ephys data (a SortGroup entry)
all_hex_sessions = sorted(set(HexMazeBlock.fetch("nwb_file_name")))
files = sorted(
    f for f in all_hex_sessions if len(sort_group_table(f) & {"nwb_file_name": f}) > 0
)

# Expand each ephys session into one (session, run interval) job per run epoch.
# This is the unit of work for the whole pipeline below.
jobs = [
    {"nwb_file_name": f, "interval_list_name": iv}
    for f in files
    for iv in get_run_intervals(f)
]

print(f"Found {len(files)} hex maze sessions with ephys, {len(jobs)} epochs:")
for f in files:
    intervals = [j["interval_list_name"] for j in jobs if j["nwb_file_name"] == f]
    print(f"  {f}: {intervals}")

# What's already populated
print(f"\nHexMazeThetaV1 already has {len(HexMazeThetaV1())} entries:")
display(HexMazeThetaV1())

Found 41 hex maze sessions with ephys, 86 epochs:
  BraveLu20240617_.nwb: ['01_r1', '03_r2', '05_r3', '07_r4']
  BraveLu20240622_.nwb: ['01_r1', '03_r2', '05_r3']
  IM-1478_20220719_.nwb: ['00_r1']
  IM-1478_20220720_.nwb: ['00_r1']
  IM-1478_20220724_.nwb: ['00_r1']
  IM-1478_20220725_.nwb: ['00_r1']
  IM-1478_20220726_.nwb: ['00_r1']
  IM-1478_20220727_.nwb: ['00_r1']
  IM-1594_20230725_.nwb: ['00_r1']
  IM-1594_20230726_.nwb: ['00_r1']
  IM-1594_20230727_.nwb: ['00_r1']
  IM-1594_20230728_.nwb: ['00_r1']
  IM-1871_20250731_.nwb: ['00_r1']
  IM-1871_20250801_.nwb: ['00_r1']
  IM-1871_20250802_.nwb: ['00_r1']
  IM-1871_20250803_.nwb: ['00_r1']
  IM-1871_20250804_.nwb: ['00_r1']
  IM-1871_20250805_.nwb: ['00_r1']
  IM-1871_20250806_.nwb: ['00_r1']
  IM-1871_20250807_.nwb: ['00_r1']
  IM-1941_20260207_.nwb: ['00_r1']
  IM-1947_20260401_.nwb: ['00_r1']
  IM-1947_20260402_.nwb: ['00_r1']
  IM-1947_20260403_.nwb: ['00_r1']
  IM-1947_20260404_.nwb: ['00_r1']
  IM-1947_20260405_.nwb: ['00_r1

lfp_merge_id,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter,nwb_file_name name of the NWB file,target_interval_list_name descriptive name of this interval list,lfp_band_sampling_rate the sampling rate for this band,analysis_file_name name of the file,analytic_signal_object_id real+imag parts of Hilbert transform,"theta_phase_object_id instantaneous phase [0, 2π] in radians",theta_power_object_id instantaneous power (amplitude²)
109fcddb-9d89-8211-dd47-b54d11c9b050,Theta 5-11 Hz,1000,Lily20251221_.nwb,03_r2,1000,Lily20251221_NJ7WFZ2HMM.nwb,368ba94c-f3cf-4f32-92f9-44d8b27c0905,01c873f0-4c9b-4cec-964a-be46197d7a71,3a7c6f1c-a64b-48a2-a0fc-42f94e91c118
147003e5-a74d-7b86-c089-878364734702,Theta 5-11 Hz,1000,Lily20251221_.nwb,01_r1,1000,Lily20251221_CGUL44UBXK.nwb,fbef0061-6f27-4755-a914-7572a7c25d0f,46931a70-72b0-40c8-a0d1-f951bc13d77b,566e0904-cb6f-42b5-b1fa-8408f6bea445
15f98b82-6683-657e-6e4c-589b020c2632,Theta 5-11 Hz,1000,Toby20250318_.nwb,07_r4_noPreTrialTimes,100,Toby20250318_A8P6GR5X10.nwb,a2cb64b9-8bd7-40af-a244-bd9c0fd15769,56ca36b6-ee74-40bb-a51a-19f9547ff903,e8181638-0b9d-41b7-bf7e-536e2521f0f3
168086c2-9629-3866-120b-8ff37b58d07b,Theta 5-11 Hz,1000,IM-1871_20250806_.nwb,00_r1,1000,IM-1871_20250806_21LWAV00JW.nwb,f8efe13c-c9dd-4bd0-9ba3-ca7194c258fd,e9fb51c9-564c-4311-a63c-c210b63eb2c6,45f6cb3f-bf9e-49ec-b970-d8324d206696
171af025-8fa5-fe7a-ca10-39e441b9878b,Theta 5-11 Hz,1000,Lily20251217_.nwb,09_r5,1000,Lily20251217_A65TQFMAT2.nwb,fd3f39aa-9520-4007-85a0-3e00f2045bde,8996d2bb-465f-4559-9862-7eabe0a64b10,b0971d13-e5b0-4baa-94d7-f32775283bdb
1981a6c7-c84e-d4d7-21e7-f51dde7cb167,Theta 5-11 Hz,1000,Lily20251217_.nwb,05_r3,1000,Lily20251217_9ZJULQGXKU.nwb,4f18ad0a-dd03-424e-b524-b4747e872013,9fbd426a-ccdd-4086-9dba-32fe3619490f,45ef09a4-515a-440f-b49b-114d70055023
236edb91-f3f0-34bf-eba7-363f9969a6c5,Theta 5-11 Hz,1034,IM-1594_20230725_.nwb,00_r1,1034,IM-1594_20230725_ZJNDWOFQ1H.nwb,73a5a509-c2d5-43d7-b977-37c06a7e44cf,5c73d063-6fc1-494b-b7fd-15362e5f83d0,3bf9bdf9-b8bf-47f9-a07d-7f7482289f09
23be3673-10cf-6c31-a0cb-06c73e0f0235,Theta 5-11 Hz,1000,BraveLu20240617_.nwb,05_r3,1000,BraveLu20240617_JG38KYQR7L.nwb,3f4ac7b6-a5ab-411f-a1d5-d89b6db73399,1b8d3668-659f-448c-949f-fc378326fe2c,e79765d4-e4ef-4ac7-8ef9-d43871c174e2
24f20c68-7544-a493-612f-85c875ea3428,Theta 5-11 Hz,1034,IM-1871_20250803_.nwb,00_r1,1034,IM-1871_20250803_OVYOCWOQ5S.nwb,6c1f231a-9385-4c3f-9816-48cdfd9cb771,f9051f1c-cae8-4d50-88a5-57c27026880b,2e8b8bbc-66f6-46b4-bc1c-026e64100684
28391cfb-3268-d0a9-32bb-7d77d1c6f2f1,Theta 5-11 Hz,1000,Toby20250316_.nwb,01_r1,1000,Toby20250316_6OFEFZLQ8V.nwb,e7a4c709-5f7f-435e-863f-34fad24a15ac,1221c299-2b25-42f0-88d5-8514eeaa54a9,ed5d41d6-c066-49cf-aef9-fdf8fb6f78ed


## Helper to get electrode IDs for a session

Uses all electrodes from sort groups

In [3]:
def get_electrode_ids(nwb_file_name):
    """Get electrode IDs for LFP processing from sort groups (excludes bad channels).

    Uses the v1 or v0 SortGroup table depending on the session (see sort_group_table).
    """
    # IM-* -> spikesorting v1, Frank lab -> spikesorting v0
    SortGroupTable = sort_group_table(nwb_file_name)
    electrodes_df = pd.DataFrame(
        (
            SortGroupTable * SortGroupTable.SortGroupElectrode
            & {"nwb_file_name": nwb_file_name}
        ).fetch(as_dict=True)
    )
    return electrodes_df["electrode_id"].unique().tolist()


# Confirm it works on a known session
test_file = sorted(files)[0]
test_electrodes = get_electrode_ids(test_file)
print(f"{test_file}: {len(test_electrodes)} electrodes")
print(test_electrodes)

BraveLu20240617_.nwb: 128 electrodes
[0, 1, 2, 3, 4, 5, 6, 7, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 8, 9, 10, 11, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 12, 13, 14, 15, 120, 121, 122, 123, 124, 125, 126, 127, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]


## Pipeline setup — loop through all sessions

Calls `setup_theta_pipeline` for each session not yet in `LFPBandSelection`.
The `LFPV1` populate inside takes ~2 hours per session. The loop is safe to interrupt
and restart — all inserts use `skip_duplicates`, and the recovery cell below reloads
any keys already in the database.

In [4]:
# Settings shared across all sessions (the interval list name now comes from each job)
THETA_FILTER_NAME = "Theta 5-11 Hz"
THETA_BAND_EDGES = [4, 5, 11, 12]

# Keyed by (nwb_file_name, interval_list_name) since a session can have several run epochs
lfp_band_keys = {}

# In LFPBandSelection the run interval is stored as target_interval_list_name
job_restrictions = [
    {"nwb_file_name": j["nwb_file_name"], "target_interval_list_name": j["interval_list_name"]}
    for j in jobs
]

# Recover any LFPBandSelection keys already in the DB (safe after kernel restart)
for row in (
    LFPBandSelection & job_restrictions & {"filter_name": THETA_FILTER_NAME}
).fetch("KEY", as_dict=True):
    key = (row["nwb_file_name"], row["target_interval_list_name"])
    lfp_band_keys[key] = row

print(f"Recovered {len(lfp_band_keys)} LFP band key(s) already in DB:")
for nwb, interval in sorted(lfp_band_keys):
    print(f"  {nwb}  [{interval}]")

Recovered 84 LFP band key(s) already in DB:
  BraveLu20240617_.nwb  [01_r1]
  BraveLu20240617_.nwb  [03_r2]
  BraveLu20240617_.nwb  [05_r3]
  BraveLu20240617_.nwb  [07_r4]
  BraveLu20240622_.nwb  [01_r1]
  BraveLu20240622_.nwb  [03_r2]
  BraveLu20240622_.nwb  [05_r3]
  IM-1478_20220719_.nwb  [00_r1]
  IM-1478_20220720_.nwb  [00_r1]
  IM-1478_20220724_.nwb  [00_r1]
  IM-1478_20220725_.nwb  [00_r1]
  IM-1478_20220726_.nwb  [00_r1]
  IM-1478_20220727_.nwb  [00_r1]
  IM-1594_20230725_.nwb  [00_r1]
  IM-1594_20230726_.nwb  [00_r1]
  IM-1594_20230727_.nwb  [00_r1]
  IM-1594_20230728_.nwb  [00_r1]
  IM-1871_20250731_.nwb  [00_r1]
  IM-1871_20250801_.nwb  [00_r1]
  IM-1871_20250802_.nwb  [00_r1]
  IM-1871_20250803_.nwb  [00_r1]
  IM-1871_20250804_.nwb  [00_r1]
  IM-1871_20250805_.nwb  [00_r1]
  IM-1871_20250806_.nwb  [00_r1]
  IM-1871_20250807_.nwb  [00_r1]
  IM-1941_20260207_.nwb  [00_r1]
  IM-1947_20260401_.nwb  [00_r1]
  IM-1947_20260402_.nwb  [00_r1]
  IM-1947_20260403_.nwb  [00_r1]
  IM-1

### Run setup_theta_pipeline for each session not yet in LFPBandSelection

In [ ]:
todo = [
    j for j in jobs
    if (j["nwb_file_name"], j["interval_list_name"]) not in lfp_band_keys
]
print(f"{len(lfp_band_keys)} job(s) already have LFPBandSelection keys.")
print(f"{len(todo)} job(s) still need setup_theta_pipeline:")
for j in todo:
    print(f"  {j['nwb_file_name']}  [{j['interval_list_name']}]")

for j in todo:
    nwb_file_name = j["nwb_file_name"]
    interval_list_name = j["interval_list_name"]
    # One LFP electrode group per file (same electrodes for all of its run epochs);
    # setup_theta_pipeline skips duplicates, so reusing the name across intervals is fine.
    lfp_group_name = nwb_file_name.replace("_.nwb", "_lfp")
    electrode_ids = get_electrode_ids(nwb_file_name)
    print(f"\nSetting up {nwb_file_name} [{interval_list_name}] ({len(electrode_ids)} electrodes)...")
    lfp_band_keys[(nwb_file_name, interval_list_name)] = HexMazeThetaV1.setup_theta_pipeline(
        nwb_file_name=nwb_file_name,
        lfp_electrode_group_name=lfp_group_name,
        electrode_ids=electrode_ids,
        interval_list_name=interval_list_name,
        theta_filter_name=THETA_FILTER_NAME,
        theta_band_edges=THETA_BAND_EDGES,
    )
    print(f"  Done. Key: {lfp_band_keys[(nwb_file_name, interval_list_name)]}")

print(f"\n{len(lfp_band_keys)} job(s) ready for LFPBandV1 populate.")

84 job(s) already have LFPBandSelection keys.
2 job(s) still need setup_theta_pipeline:
  IM-1947_20260404_.nwb  [00_r1]
  IM-1947_20260407_.nwb  [00_r1]

Setting up IM-1947_20260404_.nwb [00_r1] (201 electrodes)...


[19:57:55][INFO] Spyglass: Successfully created/updated LFPElectrodeGroup IM-1947_20260404_.nwb, IM-1947_20260404_lfp with 201 electrodes.
INFO:spyglass:Successfully created/updated LFPElectrodeGroup IM-1947_20260404_.nwb, IM-1947_20260404_lfp with 201 electrodes.
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-optogenetics': 
 ndx-optogenetics defines ExcitationSource.model as a link to ExcitationSourceModel while the core schema defines it as a link to DeviceModel. ExcitationSourceModel is not a subtype of DeviceModel. 
ndx-optogenetics defines OpticalFiber.model as a link to OpticalFiberModel while the core schema defines it as a link to DeviceModel. OpticalFiberModel is not a subtype of DeviceModel.  
This may cause compatibility issues. Please update the extension version if possible or install an older version of the core schema that is compatible.
  self._check_namespace_co

  Done. Key: {'lfp_merge_id': UUID('4dfc0d5a-8782-8eda-1453-e3c345085052'), 'filter_name': 'Theta 5-11 Hz', 'filter_sampling_rate': 1034, 'nwb_file_name': 'IM-1947_20260404_.nwb', 'target_interval_list_name': '00_r1', 'lfp_band_sampling_rate': 1034}

Setting up IM-1947_20260407_.nwb [00_r1] (195 electrodes)...


[20:16:44][INFO] Spyglass: Successfully created/updated LFPElectrodeGroup IM-1947_20260407_.nwb, IM-1947_20260407_lfp with 195 electrodes.
INFO:spyglass:Successfully created/updated LFPElectrodeGroup IM-1947_20260407_.nwb, IM-1947_20260407_lfp with 195 electrodes.
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-optogenetics': 
 ndx-optogenetics defines ExcitationSource.model as a link to ExcitationSourceModel while the core schema defines it as a link to DeviceModel. ExcitationSourceModel is not a subtype of DeviceModel. 
ndx-optogenetics defines OpticalFiber.model as a link to OpticalFiberModel while the core schema defines it as a link to DeviceModel. OpticalFiberModel is not a subtype of DeviceModel.  
This may cause compatibility issues. Please update the extension version if possible or install an older version of the core schema that is compatible.
  self._check_namespace_co

## Populate LFPBandV1 and HexMazeThetaV1

Once all setup cells above have finished, populate the theta-band LFP and then the final table.
These are much faster than `LFPV1` — a few minutes each.

In [ ]:
# Populate LFPBandV1 for all jobs whose setup is complete
# (safe to re-run — already-populated entries are skipped)
for (nwb_file, interval), lfp_band_key in sorted(lfp_band_keys.items()):
    print(f"Populating LFPBandV1 for {nwb_file} [{interval}]...")
    LFPBandV1().populate(lfp_band_key)
    print(f"  done.")

display(LFPBandV1() & [{"nwb_file_name": f} for f in files])

Populating LFPBandV1 for BraveLu20240617_.nwb [01_r1]...
  done.
Populating LFPBandV1 for BraveLu20240617_.nwb [03_r2]...
  done.
Populating LFPBandV1 for BraveLu20240617_.nwb [05_r3]...
  done.
Populating LFPBandV1 for BraveLu20240617_.nwb [07_r4]...
  done.
Populating LFPBandV1 for BraveLu20240622_.nwb [01_r1]...
  done.
Populating LFPBandV1 for BraveLu20240622_.nwb [03_r2]...
  done.
Populating LFPBandV1 for BraveLu20240622_.nwb [05_r3]...
  done.
Populating LFPBandV1 for IM-1478_20220719_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1478_20220720_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1478_20220724_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1478_20220725_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1478_20220726_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1478_20220727_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1594_20230725_.nwb [00_r1]...
  done.
Populating LFPBandV1 for IM-1594_20230726_.nwb [00_r1]...
  done.
Populating LFPBan

lfp_merge_id,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter,nwb_file_name name of the NWB file,target_interval_list_name descriptive name of this interval list,lfp_band_sampling_rate the sampling rate for this band,analysis_file_name name of the file,interval_list_name descriptive name of this interval list,lfp_band_object_id the NWB object ID for loading this object from the file
0416029d-afde-7967-6820-fdd6f81f3f50,Ripple 150-250 Hz,1034,IM-1871_20250803_.nwb,00_r1,1034,IM-1871_20250803_DVFZB1UDH9.nwb,00_r1 lfp band 1034Hz,e8a8869d-1b3a-4a91-9053-5178354fd6f1
109fcddb-9d89-8211-dd47-b54d11c9b050,Theta 5-11 Hz,1000,Lily20251221_.nwb,03_r2,1000,Lily20251221_PC5YFWTFPQ.nwb,03_r2 lfp band 1000Hz,d11f271a-49e0-414b-a005-7636cf24b085
147003e5-a74d-7b86-c089-878364734702,Theta 5-11 Hz,1000,Lily20251221_.nwb,01_r1,1000,Lily20251221_JNHRFT52OP.nwb,01_r1 lfp band 1000Hz,d91026f0-6893-420a-bfb6-261c1581da76
14892619-7346-a8b1-30ea-63465f4876cf,Ripple 150-250 Hz,1034,IM-1871_20250804_.nwb,00_r1,1034,IM-1871_20250804_DWMMQ0B4WV.nwb,00_r1 lfp band 1034Hz,2a91a2da-e477-4c41-895d-a0efed53a2db
15f98b82-6683-657e-6e4c-589b020c2632,Theta 5-11 Hz,1000,Toby20250318_.nwb,07_r4_noPreTrialTimes,100,Toby20250318_H7UD8XONMP.nwb,07_r4_noPreTrialTimes lfp band 100Hz,d40788e1-3f19-486e-89b6-9504fd598f1b
168086c2-9629-3866-120b-8ff37b58d07b,Theta 5-11 Hz,1000,IM-1871_20250806_.nwb,00_r1,1000,IM-1871_20250806_6I9KMJN9YO.nwb,00_r1 lfp band 1000Hz,00434128-e79b-41c0-a604-387aa0fddb39
171af025-8fa5-fe7a-ca10-39e441b9878b,Theta 5-11 Hz,1000,Lily20251217_.nwb,09_r5,1000,Lily20251217_MC0FHPDQLU.nwb,09_r5 lfp band 1000Hz,4eb4cfea-379c-4da3-87ea-41092a47fc22
1981a6c7-c84e-d4d7-21e7-f51dde7cb167,Theta 5-11 Hz,1000,Lily20251217_.nwb,05_r3,1000,Lily20251217_9E8IKS6RYI.nwb,05_r3 lfp band 1000Hz,43533d5b-54d3-48c7-8cff-a8a69ef34c21
1d2d7af5-8758-4103-e367-9985abc3a67e,Ripple 150-250 Hz,1034,IM-1871_20250805_.nwb,00_r1,1034,IM-1871_20250805_1AFEHECI83.nwb,00_r1 lfp band 1034Hz,0ad2471a-e559-4cfb-90bb-e100f789f7cd
1f5ef082-1668-e949-0455-43d28f4d1ff6,Theta 5-11 Hz,1000,Toby20250316_.nwb,05_r3_noPreTrialTimes,100,Toby20250316_7CGEIWD7N6.nwb,05_r3_noPreTrialTimes lfp band 100Hz,aed6c54c-c89b-4082-be18-dc11c94bbff4


In [ ]:
# Populate HexMazeThetaV1 for all jobs
# (computes analytic signal, phase, power and saves to analysis NWB)
for (nwb_file, interval), lfp_band_key in sorted(lfp_band_keys.items()):
    print(f"Populating HexMazeThetaV1 for {nwb_file} [{interval}]...")
    HexMazeThetaV1().populate(lfp_band_key)
    print(f"  done.")

display(HexMazeThetaV1() & [{"nwb_file_name": f} for f in files])

Populating HexMazeThetaV1 for BraveLu20240617_.nwb [01_r1]...
  done.
Populating HexMazeThetaV1 for BraveLu20240617_.nwb [03_r2]...
  done.
Populating HexMazeThetaV1 for BraveLu20240617_.nwb [05_r3]...
  done.
Populating HexMazeThetaV1 for BraveLu20240617_.nwb [07_r4]...
  done.
Populating HexMazeThetaV1 for BraveLu20240622_.nwb [01_r1]...
  done.
Populating HexMazeThetaV1 for BraveLu20240622_.nwb [03_r2]...
  done.
Populating HexMazeThetaV1 for BraveLu20240622_.nwb [05_r3]...
  done.
Populating HexMazeThetaV1 for IM-1478_20220719_.nwb [00_r1]...
  done.
Populating HexMazeThetaV1 for IM-1478_20220720_.nwb [00_r1]...
  done.
Populating HexMazeThetaV1 for IM-1478_20220724_.nwb [00_r1]...
  done.
Populating HexMazeThetaV1 for IM-1478_20220725_.nwb [00_r1]...
  done.
Populating HexMazeThetaV1 for IM-1478_20220726_.nwb [00_r1]...
  done.
Populating HexMazeThetaV1 for IM-1478_20220727_.nwb [00_r1]...
  done.
Populating HexMazeThetaV1 for IM-1594_20230725_.nwb [00_r1]...
  done.
Populating He

lfp_merge_id,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter,nwb_file_name name of the NWB file,target_interval_list_name descriptive name of this interval list,lfp_band_sampling_rate the sampling rate for this band,analysis_file_name name of the file,analytic_signal_object_id real+imag parts of Hilbert transform,"theta_phase_object_id instantaneous phase [0, 2π] in radians",theta_power_object_id instantaneous power (amplitude²)
109fcddb-9d89-8211-dd47-b54d11c9b050,Theta 5-11 Hz,1000,Lily20251221_.nwb,03_r2,1000,Lily20251221_NJ7WFZ2HMM.nwb,368ba94c-f3cf-4f32-92f9-44d8b27c0905,01c873f0-4c9b-4cec-964a-be46197d7a71,3a7c6f1c-a64b-48a2-a0fc-42f94e91c118
147003e5-a74d-7b86-c089-878364734702,Theta 5-11 Hz,1000,Lily20251221_.nwb,01_r1,1000,Lily20251221_CGUL44UBXK.nwb,fbef0061-6f27-4755-a914-7572a7c25d0f,46931a70-72b0-40c8-a0d1-f951bc13d77b,566e0904-cb6f-42b5-b1fa-8408f6bea445
15f98b82-6683-657e-6e4c-589b020c2632,Theta 5-11 Hz,1000,Toby20250318_.nwb,07_r4_noPreTrialTimes,100,Toby20250318_A8P6GR5X10.nwb,a2cb64b9-8bd7-40af-a244-bd9c0fd15769,56ca36b6-ee74-40bb-a51a-19f9547ff903,e8181638-0b9d-41b7-bf7e-536e2521f0f3
168086c2-9629-3866-120b-8ff37b58d07b,Theta 5-11 Hz,1000,IM-1871_20250806_.nwb,00_r1,1000,IM-1871_20250806_21LWAV00JW.nwb,f8efe13c-c9dd-4bd0-9ba3-ca7194c258fd,e9fb51c9-564c-4311-a63c-c210b63eb2c6,45f6cb3f-bf9e-49ec-b970-d8324d206696
171af025-8fa5-fe7a-ca10-39e441b9878b,Theta 5-11 Hz,1000,Lily20251217_.nwb,09_r5,1000,Lily20251217_A65TQFMAT2.nwb,fd3f39aa-9520-4007-85a0-3e00f2045bde,8996d2bb-465f-4559-9862-7eabe0a64b10,b0971d13-e5b0-4baa-94d7-f32775283bdb
1981a6c7-c84e-d4d7-21e7-f51dde7cb167,Theta 5-11 Hz,1000,Lily20251217_.nwb,05_r3,1000,Lily20251217_9ZJULQGXKU.nwb,4f18ad0a-dd03-424e-b524-b4747e872013,9fbd426a-ccdd-4086-9dba-32fe3619490f,45ef09a4-515a-440f-b49b-114d70055023
236edb91-f3f0-34bf-eba7-363f9969a6c5,Theta 5-11 Hz,1034,IM-1594_20230725_.nwb,00_r1,1034,IM-1594_20230725_ZJNDWOFQ1H.nwb,73a5a509-c2d5-43d7-b977-37c06a7e44cf,5c73d063-6fc1-494b-b7fd-15362e5f83d0,3bf9bdf9-b8bf-47f9-a07d-7f7482289f09
23be3673-10cf-6c31-a0cb-06c73e0f0235,Theta 5-11 Hz,1000,BraveLu20240617_.nwb,05_r3,1000,BraveLu20240617_JG38KYQR7L.nwb,3f4ac7b6-a5ab-411f-a1d5-d89b6db73399,1b8d3668-659f-448c-949f-fc378326fe2c,e79765d4-e4ef-4ac7-8ef9-d43871c174e2
24f20c68-7544-a493-612f-85c875ea3428,Theta 5-11 Hz,1034,IM-1871_20250803_.nwb,00_r1,1034,IM-1871_20250803_OVYOCWOQ5S.nwb,6c1f231a-9385-4c3f-9816-48cdfd9cb771,f9051f1c-cae8-4d50-88a5-57c27026880b,2e8b8bbc-66f6-46b4-bc1c-026e64100684
28391cfb-3268-d0a9-32bb-7d77d1c6f2f1,Theta 5-11 Hz,1000,Toby20250316_.nwb,01_r1,1000,Toby20250316_6OFEFZLQ8V.nwb,e7a4c709-5f7f-435e-863f-34fad24a15ac,1221c299-2b25-42f0-88d5-8514eeaa54a9,ed5d41d6-c066-49cf-aef9-fdf8fb6f78ed


## Sanity check — fetch and plot for one session

In [ ]:
import matplotlib.pyplot as plt

# Pick a session
check_nwb = "IM-1478_20220725_.nwb"
theta_entry = HexMazeThetaV1() & {"nwb_file_name": check_nwb}

phase_df = theta_entry.fetch1_theta_phase()
power_df = theta_entry.fetch1_theta_power()

print(f"Theta phase shape: {phase_df.shape}  (n_timesteps × n_electrodes)")
print(f"Sampling rate: {1 / np.median(np.diff(phase_df.index)):.1f} Hz")
display(phase_df.head())
display(power_df.head())

[2026-07-15 07:33:38,397][WARNING]: Skipped checksum for file with hash: e579349a-10f1-9245-8daf-740f093ecb2a, and path: /stelmo/nwb/analysis/IM-1478_20220725/IM-1478_20220725_IMSEN7F3KS.nwb
[2026-07-15 07:33:38,408][WARNING]: Skipped checksum for file with hash: e579349a-10f1-9245-8daf-740f093ecb2a, and path: /stelmo/nwb/analysis/IM-1478_20220725/IM-1478_20220725_IMSEN7F3KS.nwb
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-optogenetics': 
 ndx-optogenetics defines ExcitationSource.model as a link to ExcitationSourceModel while the core schema defines it as a link to DeviceModel. ExcitationSourceModel is not a subtype of DeviceModel. 
ndx-optogenetics defines OpticalFiber.model as a link to OpticalFiberModel while the core schema defines it as a link to DeviceModel. OpticalFiberModel is not a subtype of DeviceModel.  
This may cause compatibility issues. Please update the extens

Theta phase shape: (4863702, 166)  (n_timesteps × n_electrodes)
Sampling rate: 1000.0 Hz


,electrode 56,electrode 57,electrode 58,electrode 59,electrode 60,electrode 61,electrode 62,electrode 63,electrode 65,electrode 66,...,electrode 245,electrode 246,electrode 247,electrode 249,electrode 250,electrode 251,electrode 252,electrode 253,electrode 254,electrode 255
time,,,,,,,,,,,,,,,,,,,,,
12.51145,1.803954,1.800348,1.795746,1.804022,1.808900,1.813796,1.840642,1.830888,1.811792,1.807629,...,4.932505,4.957762,4.940413,4.961965,4.962997,4.967390,4.968818,4.954723,4.977185,4.923812
12.51245,2.014603,2.004534,2.012398,2.023029,2.034072,2.039138,2.091155,2.080769,2.027024,2.018510,...,5.132389,5.183035,5.151480,5.190880,5.193172,5.197129,5.203116,5.178955,5.225401,5.109177
12.51345,2.038214,2.022581,2.031728,2.043386,2.047981,2.060242,2.107929,2.100635,2.050359,2.038523,...,5.144927,5.190217,5.159825,5.196642,5.208997,5.210786,5.212489,5.182513,5.232822,5.109343
12.51445,2.209875,2.199929,2.217119,2.218400,2.236254,2.247493,2.309208,2.297232,2.245212,2.223309,...,5.322615,5.384737,5.348518,5.398500,5.404338,5.406505,5.414885,5.386965,5.454922,5.275035
12.51545,2.237418,2.213537,2.228030,2.244126,2.261159,2.278617,2.331406,2.321190,2.261084,2.238590,...,5.336228,5.399613,5.356945,5.420271,5.416862,5.424143,5.431996,5.397783,5.465363,5.266389


,electrode 56,electrode 57,electrode 58,electrode 59,electrode 60,electrode 61,electrode 62,electrode 63,electrode 65,electrode 66,...,electrode 245,electrode 246,electrode 247,electrode 249,electrode 250,electrode 251,electrode 252,electrode 253,electrode 254,electrode 255
time,,,,,,,,,,,,,,,,,,,,,
12.51145,21654.132429,23660.154219,21887.419646,22933.609489,24608.931101,27635.848459,29774.963460,31994.602629,17976.505479,19782.432879,...,85917.324274,113948.652040,101447.905681,30308.592398,64536.057407,62377.852841,97019.745498,113935.441193,154890.315374,68691.510937
12.51245,7029.448625,7750.964745,7095.142793,7562.312199,8011.907147,9074.041646,9712.823685,10491.240320,5610.580060,6536.562030,...,26198.561545,34314.376005,29486.908105,9131.850958,19753.890080,18863.769335,28816.833432,31629.962669,44041.136261,19524.609178
12.51345,7112.448762,7980.289580,7299.127927,7721.976382,8362.623282,9573.441187,10328.058606,10997.430757,5754.355350,6734.768710,...,26316.301666,34166.698238,28468.250276,9343.239217,19775.275254,19063.372138,28534.397870,30413.290135,42076.559883,18090.827053
12.51445,4275.351594,4854.653610,4411.224834,4846.912853,5312.082408,6122.613820,6676.482454,7108.300842,3323.705967,4125.603947,...,14921.305427,19512.249257,15098.971476,5503.730146,11358.550734,10969.549355,16106.259576,15596.666399,22312.518697,8785.994727
12.51545,4396.641750,4909.952239,4503.873490,4978.563150,5447.117208,6396.471543,7078.801969,7485.637861,3376.392214,4172.352944,...,14772.705769,19241.866663,14359.588867,5449.462320,11351.192514,11159.774262,15858.721497,14798.019310,20960.284861,7981.229464
